## tl;dr

The complete robust sweep confirms 97%+ success before 50k: run A reaches 97.65% at 26k and 30k, while run B reaches 97.85% at 28k. Both use global batch 128.

## Context & Methods

Parse only complete summaries from the saved A/B checkpoint sweeps. A valid point must contain all four suites, 40 tasks, and its associated evaluation has 2,000 episode rows (50 episodes per task).

### Key Assumptions
A and B are independent training runs and are never concatenated into one trajectory.

## Data

In [1]:
from pathlib import Path
import csv

report_dir = Path('/root/feihong/starVLA/reports/libero_stage2_early_eval_20260812')
rows = list(csv.DictReader((report_dir / 'early_eval_curve.csv').open()))
len(rows), sorted(set(row['run'] for row in rows))

(27, ['A: LR 2.5e-5 / warmup 3k', 'B: LR 2.0e-5 / warmup 5k'])

## Results

In [2]:
for run in sorted(set(row['run'] for row in rows)):
    run_rows = [row for row in rows if row['run'] == run]
    best = max(run_rows, key=lambda row: float(row['overall_success']))
    print(run, 'best_step=', best['step'], 'overall=', float(best['overall_success']), 'libero_10=', float(best['libero_10_success']))

a30 = next(row for row in rows if row['run'].startswith('A:') and row['step'] == '30000')
assert float(a30['overall_success']) == 0.9765
assert int(a30['global_batch']) == 128
print('A@30k verified:', a30['overall_success'], 'global_batch=', a30['global_batch'])

A: LR 2.5e-5 / warmup 3k best_step= 26000 overall= 0.9765 libero_10= 0.9520000000000001
B: LR 2.0e-5 / warmup 5k best_step= 28000 overall= 0.9784999999999999 libero_10= 0.958
A@30k verified: 0.9765 global_batch= 128


## Takeaways

Keep global batch 128 for the new weighted-Stage1 experiment. The higher-value change is to retain and evaluate checkpoints densely from 26k to 40k, preserving the Stage1 change as the main experimental variable.